In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import  GradientBoostingClassifier

# load data train test split
heart_df = pd.read_csv("../data/heart_2020_cleaned.csv")
# for now drop a lot of other values so there is no need to switch up the UI
X = heart_df.drop(['HeartDisease'], axis=1)  
y = heart_df['HeartDisease']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Preprocessor
X_num = X_train.select_dtypes(include='number')
X_cat = X_train.select_dtypes(exclude='number')
# Categorical columns
cat_cols = X_cat.columns

# Numeric columns (all others)
num_cols = X_num.columns

# Numeric pipeline
num_pipe = Pipeline([
    ("scaler", StandardScaler())
])

# Categorical pipeline
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combine into preprocessor
preprocessor = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
])

model = GradientBoostingClassifier(
    n_estimators=135,                # Moderate number of trees
    learning_rate=0.05040612177770395,# Standard learning rate
    max_depth=9,                     # Shallow trees (typical for boosting)
    min_samples_split=1040,
    min_samples_leaf=1444,
    subsample=0.9771414282231924,     # Use 80% of data per tree (faster + regularization)
    max_features='sqrt',
    random_state=42,
    verbose=0                        # Show progress
)
pipe = Pipeline([
    ("prep", preprocessor),
    ("model", model)  # No feature selector - boosting handles this
])
pipe.fit(X_train, y_train)

In [ ]:
# store the trained pipeline
import pickle
pickle.dump(pipe,
            open(file='../models/trained_pipe_gradBoost.sav',
                 mode='wb'))